# 02 — Exploratory Data Analysis

**Objective:** perform reproducible EDA on the shared training partition.  
**Owner:** Member 02  

> Leakage warning: the reserved test set is not inspected.

In [ ]:
from pathlib import Path
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'configs/config.yaml').is_file())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import load_config
from src.data import audit_numeric_features, get_dataset_summary, load_dataset
from src.validation import create_train_test_split
from src.visualization import save_figure

config = load_config()
X_all, y_all, metadata = load_dataset(optimize_memory=True)
X_train, X_test, y_train, y_test = create_train_test_split(X_all, y_all)
X, y = X_train, y_train  # Later EDA cells see training data only.
print(config['project']['name'])
print('Training shape:', X.shape)
print('Reserved test shape (not inspected):', X_test.shape)

## Shared structural audit

In [ ]:
train_metadata = {**metadata, 'partition': 'train'}
dataset_summary = get_dataset_summary(X, y, train_metadata)
feature_audit = audit_numeric_features(X)
summary_fields = [
    'n_rows', 'n_features', 'numeric_feature_count', 'missing_values_X',
    'constant_feature_count', 'quasi_constant_feature_count', 'infinity_count',
    'target_value_counts', 'target_proportions', 'total_memory_mb',
]
display(pd.Series({field: dataset_summary[field] for field in summary_fields}, name='value'))
display(feature_audit.head())

## Feature distributions

In [ ]:
display(X.describe().T)
features_to_plot = ['var_0', 'var_1', 'var_2', 'var_3', 'var_4']
fig, axes = plt.subplots(len(features_to_plot), 1, figsize=(8, 14))
for ax, feature in zip(axes, features_to_plot):
    ax.hist(X[feature], bins=40)
    ax.set(title=f'Distribution of {feature}', xlabel=feature, ylabel='Frequency')
fig.tight_layout()
save_figure(fig, project_root / 'reports/figures/eda_feature_distributions.pdf')
plt.show()

## Differences by target class

In [ ]:
class_means = X.groupby(y, observed=True).mean().T
class_means.columns = [f'target_{label}_mean' for label in class_means.columns]
mean_difference = class_means.iloc[:, 1] - class_means.iloc[:, 0]
class_means['absolute_difference'] = mean_difference.abs()
class_means['standardized_difference'] = mean_difference.abs().div(X.std()).replace([np.inf, -np.inf], np.nan)
class_means = class_means.sort_values('standardized_difference', ascending=False)
display(class_means.head(15))

In [ ]:
top_features = class_means.head(3).index
target_labels = list(pd.unique(y))
fig, axes = plt.subplots(1, len(top_features), figsize=(16, 4))
for ax, feature in zip(axes, top_features):
    for label in target_labels:
        ax.hist(X.loc[y == label, feature], bins=40, alpha=0.5, density=True, label=f'Target = {label}')
    ax.set(title=f'{feature} by target', xlabel=feature, ylabel='Density')
    ax.legend()
fig.tight_layout()
save_figure(fig, project_root / 'reports/figures/eda_top_features_by_target.pdf')
plt.show()

## Correlations and potential outliers

In [ ]:
correlation = X.corr()
upper_triangle = np.triu(np.ones(correlation.shape, dtype=bool), k=1)
correlation_pairs = correlation.where(upper_triangle).stack().abs().sort_values(ascending=False)
display(correlation_pairs.head(20).rename('absolute_correlation'))

q1, q3 = X.quantile(0.25), X.quantile(0.75)
iqr = q3 - q1
outlier_summary = ((X.lt(q1 - 1.5 * iqr)) | (X.gt(q3 + 1.5 * iqr))).sum().sort_values(ascending=False)
display(outlier_summary.head(20).rename('potential_outlier_count'))

## EDA summary

- Shared loading, memory optimization, auditing, splitting, and figure saving utilities are used.
- Structural claims should be read from the executed `dataset_summary` and `feature_audit`.
- Class differences, correlations, and IQR flags are descriptive—not causal findings or automatic reasons to remove features.
- The reserved final test partition remains untouched.